In [11]:
import numpy as np
import pandas as pd
from PIL import Image
import xarray as xr
import matplotlib.pyplot as plt

In [ ]:
# IME Funcs
m_ch4 = 0.01614  # mass of methane kg/mol


def get_IME(enh_img, mask, a=25**2):
    delta_x = enh_img * mask  # enh_img: mol/m^2
    IME = np.abs(np.sum(delta_x * m_ch4 * a))  # kg
    return IME


def get_ueff(u10, v10):
    # This is the approach use by varon et al but may need to recalibrate
    u10 = np.mean(u10)
    v10 = np.mean(v10)
    ueff = (u10**2 + v10**2) ** (1 / 2)
    # u_eff = 0.23*ueff+0.7
    # ueff = np.log(ueff)+0.5
    return ueff


# calculate source rate
def get_source_rate(img, mask, u10, v10, a=20**2):
    L = (np.sum(mask) * a) ** (
        1 / 2
    )  # a: pixel resolution  L:square root of the plume area
    IME = get_IME(img, mask, a)  # IME
    u_eff = get_ueff(u10, v10)  # Ueff
    if L == 0:
        return 0, 0, 0, 0
    Q = u_eff / L * IME / (10**3) * 3600  # t/h    #Source Rate:t/h
    return L, IME, u_eff, Q

In [ ]:
def cal_emission_rate(
    wind_file_path, time_index, tar_lat, tar_lon, emis_image, mask_image, size
):
    data = xr.open_dataset(wind_file_path)

    # Extraction
    lon = data["longitude"]
    lat = data["latitude"]
    v10 = data["v10"][time_index, :, :]
    u10 = data["u10"][time_index, :, :]
    # time = data["valid_time"][time_index]

    # find the closest location
    diff_lat = [abs(x - tar_lat) for x in lat]
    lat_index = diff_lat.index(min(diff_lat))
    diff_lon = [abs(x - tar_lon) for x in lon]
    lon_index = diff_lon.index(min(diff_lon))

    # adjust the image size
    mid_loc_x = emis_image.shape[0] // 2
    mid_loc_y = emis_image.shape[1] // 2
    emis_image = emis_image[
        mid_loc_x - size : mid_loc_x + size, mid_loc_y - size : mid_loc_y + size
    ]
    mask_image = mask_image[
        mid_loc_x - size : mid_loc_x + size, mid_loc_y - size : mid_loc_y + size
    ]
    # print(emis_image.shape, mask_image.shape)

    ll, ime, ueff, er = get_source_rate(
        emis_image,
        mask_image,
        u10[lat_index, lon_index].values,
        v10[lat_index, lon_index].values,
    )  # t/h
    return ll, ime, ueff, er

#### Physical Model

In [ ]:
all_data = pd.read_csv("data_details.csv")
all_data["emission_rate"], all_data["L"], all_data["IME"], all_data["Ueff"] = 0, 0, 0, 0
plume_data = all_data[all_data["plume"] == 2]
for index, data in plume_data.iterrows():
    subname = data["index"]
    wind_file_path = "./dataset/wind/" + str(data["index"]) + ".nc"
    time_index = 18
    lat, lng = data["latitude"], data["longitude"]
    emis_image = np.load("./dataset/mbmp/{}.npy".format(subname))
    mask_image = (
        np.array(Image.open("./dataset/mask_gray/{}.jpg".format(subname)).convert("L"))
        / 255
    )
    size = 80
    l2, ime, ueff, er = cal_emission_rate(
        wind_file_path, time_index, lat, lng, emis_image, mask_image, size
    )
    all_data.loc[index, "emission_rate"] = er
    all_data.loc[index, "L"] = l2
    all_data.loc[index, "IME"] = ime
    all_data.loc[index, "Ueff"] = ueff
    print(index, l2, ime, ueff, er)

In [103]:
all_data.to_csv("./dataset/data_details_new.csv", index=False)

### Test

In [ ]:
all_data = pd.read_csv("./dataset/data_details_new.csv")
all_data[all_data["index"] == 2504]

In [ ]:
for index, data in all_data.iterrows():
    if index != 2504:
        continue
    subname = data["index"]
    wind_file_path = "./dataset/wind/" + str(data["index"]) + ".nc"
    time_index = 18
    lat, lng = data["latitude"], data["longitude"]
    emis_image = np.load("./dataset/mbmp/{}.npy".format(subname))
    mask_image = (
        np.array(Image.open("./dataset/mask_gray/{}.jpg".format(subname)).convert("L"))
        / 255
    )
    size = 80
    er = cal_emission_rate(
        wind_file_path, time_index, lat, lng, emis_image, mask_image, size
    )
    print(index, er)

In [ ]:
plt.imshow(emis_image)
plt.colorbar()
plt.show()

In [ ]:
plt.imshow(mask_image)
plt.show()

In [ ]:
all_data[all_data["emission_rate"] > 0]

In [ ]:
train_q = pd.read_csv("train_q.csv")
train_q = train_q.merge(
    all_data[["index", "emission_rate", "L", "IME", "Ueff"]], on="index", how="left"
)
train_q.to_csv("train_r.csv", index=False)

In [ ]:
val_q = pd.read_csv("val_q.csv")
val_q = val_q.merge(
    all_data[["index", "emission_rate", "L", "IME", "Ueff"]], on="index", how="left"
)
val_q.to_csv("val_r.csv", index=False)

In [ ]:
test_q = pd.read_csv("test_q.csv")
test_q = test_q.merge(
    all_data[["index", "emission_rate", "L", "IME", "Ueff"]], on="index", how="left"
)
test_q.to_csv("test_r.csv", index=False)

In [53]:
train_r = pd.read_csv("train_r.csv")
pos = train_r[train_r["plume"] == 2][~train_r["site"].isin(["U5", "U8"])]
neg = train_r[train_r["plume"] == 0][~train_r["site"].isin(["U5", "U8"])]

/tmp/ipykernel_225246/3340131561.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  pos = train_r[train_r['plume']==2][~train_r['site'].isin(['U5','U8'])]
/tmp/ipykernel_225246/3340131561.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  neg = train_r[train_r['plume']==0][~train_r['site'].isin(['U5','U8'])]


In [ ]:
neg[:100]

### Others

In [ ]:
data = xr.open_dataset("./dataset/wind/10034.nc")
data["v10"][18, :, :].shape, data["u10"][18, :, :].shape

### Predictions

In [ ]:
all_data = pd.read_csv("data_details.csv")
plume_data = all_data[all_data["plume"] == 2]
pre_ers, gt_ers = [], []
for index, data in plume_data.iterrows():
    subname = data["index"]
    if subname == 338:
        continue
    wind_file_path = "./dataset/wind/" + str(data["index"]) + ".nc"
    time_index = 18
    size = 80
    lat, lng = data["latitude"], data["longitude"]
    emis_image = np.load("./dataset/mbmp_p/{}_p.npy".format(subname))
    mask_image = np.load("./dataset/mask_gray_p/{}_p.npy".format(subname))
    pre_er = cal_emission_rate(
        wind_file_path, time_index, lat, lng, emis_image, mask_image, size
    )
    emis_image = np.load("./dataset/mbmp/{}.npy".format(subname))
    mask_image = (
        np.array(Image.open("./dataset/mask_gray/{}.jpg".format(subname)).convert("L"))
        / 255
    )
    gt_er = cal_emission_rate(
        wind_file_path, time_index, lat, lng, emis_image, mask_image, size
    )
    pre_ers.append(pre_er)
    gt_ers.append(gt_er)
    # all_data.loc[index, 'emission_rate'] = er
    print(index, pre_er, gt_er)

In [ ]:
from sklearn.metrics import r2_score

r2 = r2_score(gt_ers, pre_ers)
r2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

degree = 1
coefficients = np.polyfit(gt_ers, pre_ers, degree)
polynomial = np.poly1d(coefficients)

y_pred = polynomial(gt_ers)
mse = np.mean((pre_ers - y_pred) ** 2)
print(f"Mean Squared Error (MSE): {mse}")
r2 = r2_score(pre_ers, y_pred)
print(f"R2: {r2}")

x_fit = np.linspace(min(gt_ers), max(gt_ers), 100)
y_fit = polynomial(x_fit)

plt.scatter(gt_ers, pre_ers, label="Original Data", color="blue")
plt.plot(x_fit, y_fit, label=f"Polynomial Fit (degree={degree})", color="red")

plt.plot(x_fit, x_fit, label="y = x (reference)", color="green", linestyle="--")

plt.title("Polynomial Curve Fitting")
plt.xlabel("X-axis")
plt.ylabel("Y-axis")
plt.legend()

plt.show()

In [ ]:
a = np.array([])
b = np.array([])
c = np.array([])

In [ ]:
degree = 1
coefficients = np.polyfit(a, b, degree)
polynomial = np.poly1d(coefficients)

y_pred = polynomial(a)
mse = np.mean((b - y_pred) ** 2)
print(f"Mean Squared Error (MSE): {mse}")
mse = np.mean((b - a) ** 2)
print(f"Original Mean Squared Error (MSE): {mse}")
r2 = r2_score(b, y_pred)
print(f"R2: {r2}")
r2 = r2_score(a, b)
print(f"Original R2: {r2}")

x_fit = np.linspace(min(a), max(a), 100)
y_fit = polynomial(x_fit)


plt.scatter(a, b, label="Original Data", color="blue")
plt.plot(x_fit, y_fit, label=f"Polynomial Fit (degree={degree})", color="red")

plt.plot(x_fit, x_fit, label="y = x (reference)", color="green", linestyle="--")

plt.title("Polynomial Curve Fitting")
plt.xlabel("X-axis")
plt.ylabel("Y-axis")
plt.legend()

slope = coefficients[0]
plt.text(
    0.1,
    0.9,
    f"Slope: {slope:.2f}",
    transform=plt.gca().transAxes,
    fontsize=12,
    color="black",
)

plt.show()

In [ ]:
degree = 1
coefficients = np.polyfit(a, c, degree)
polynomial = np.poly1d(coefficients)

y_pred = polynomial(a)
mse = np.mean((c - y_pred) ** 2)
print(f"Mean Squared Error (MSE): {mse}")
mse = np.mean((c - a) ** 2)
print(f"Original Mean Squared Error (MSE): {mse}")
r2 = r2_score(c, y_pred)
print(f"R2: {r2}")
r2 = r2_score(a, c)
print(f"Original R2: {r2}")


x_fit = np.linspace(min(a), max(a), 100)
y_fit = polynomial(x_fit)


plt.scatter(a, c, label="Original Data", color="blue")
plt.plot(x_fit, y_fit, label=f"Polynomial Fit (degree={degree})", color="red")

plt.plot(x_fit, x_fit, label="y = x (reference)", color="green", linestyle="--")

plt.title("Polynomial Curve Fitting")
plt.xlabel("X-axis")
plt.ylabel("Y-axis")
plt.legend()

slope = coefficients[0]
plt.text(
    0.1,
    0.9,
    f"Slope: {slope:.2f}",
    transform=plt.gca().transAxes,
    fontsize=12,
    color="black",
)

plt.show()